# Notebook 02 - Trend And Piecewise Growth Regimes

This notebook estimates the log real GDP trend and piecewise constant growth regimes.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from us_gdp_regime.config import load_config
from us_gdp_regime.pipeline import fit_models, make_figures, prepare_data
from us_gdp_regime.robustness import (
    run_criterion_sensitivity,
    run_exclusion_sensitivity,
    run_min_segment_sensitivity,
    summarize_recurring_breaks,
)
from us_gdp_regime.plotting import plot_break_sensitivity

config = load_config(Path("config/default.yaml"))
series_path = config.paths.processed_dir / "us_gdp_series.csv"
if not series_path.exists():
    prepare_data(config)
series = pd.read_csv(series_path)
series.head()

## Raw Real GDP And Log Real GDP

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(series["year"], series["real_gdp"], linewidth=1.2)
axes[0].set_title("United States real GDP proxy")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Real GDP proxy")
axes[0].grid(alpha=0.3)
axes[1].plot(series["year"], series["log_real_gdp"], linewidth=1.2)
axes[1].set_title("Log real GDP")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Log real GDP")
axes[1].grid(alpha=0.3)
fig.text(0.01, -0.02, "Source: Maddison Project Database 2023.", fontsize=9)
fig.tight_layout()

## Fit Log Real GDP Trend Regression

In [ ]:
model_outputs = fit_models(config)
trend = pd.read_csv(model_outputs["trend"])
trend

The trend slope is interpreted as an average annualised exponential growth rate. It summarizes the long-run path, but it does not imply that every period grew at that speed.

## Trend Residuals

In [ ]:
from us_gdp_regime.models import fit_log_trend

trend_result, trend_frame = fit_log_trend(series)
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(trend_frame["year"], trend_frame["trend_residual"], linewidth=1.2)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Residuals from log real GDP trend regression")
ax.set_xlabel("Year")
ax.set_ylabel("Log residual")
ax.grid(alpha=0.3)
fig.tight_layout()

## Fit Piecewise Constant Growth Regimes

Regime classification is based on annual growth rates, not GDP levels. GDP levels trend upward over long horizons, so above/below-mean labels on levels would mostly identify early versus late periods rather than growth performance.

In [ ]:
segments = pd.read_csv(model_outputs["segments"])
segments

## Selected Breakpoints

In [ ]:
break_years = segments["end_year"].iloc[:-1].tolist()
break_years

## Article Figures

In [ ]:
figure_outputs = make_figures(config)
figure_outputs

## Sensitivity Analysis

In [ ]:
min_size_results = run_min_segment_sensitivity(series, [4, 5, 7, 10], criterion="bic", max_breaks=config.model.max_breaks)
criterion_results = run_criterion_sensitivity(series, criteria=("bic", "aic"), min_segment_size=config.model.min_segment_size, max_breaks=config.model.max_breaks)
exclude_wwii = run_exclusion_sensitivity(series, [(1941, 1945)], criterion=config.model.criterion, min_segment_size=config.model.min_segment_size, max_breaks=config.model.max_breaks)
robustness = pd.concat([min_size_results, criterion_results, exclude_wwii], ignore_index=True)
robustness_path = config.paths.models_dir / "regime_robustness.csv"
robustness.to_csv(robustness_path, index=False)
robustness.head()

In [ ]:
recurring = summarize_recurring_breaks(robustness, tolerance=2)
recurring_path = config.paths.models_dir / "recurring_break_years.csv"
recurring.to_csv(recurring_path, index=False)
if not recurring.empty:
    plot_break_sensitivity(recurring, config.paths.figures_dir / "breakpoint_sensitivity.png", dpi=config.plots.dpi)
recurring

## Sensitivity Interpretation

Stable break years are those that recur across criteria, minimum segment sizes, or exclusion scenarios. Unstable break years should not be overinterpreted. The configured BIC model remains the default article model because BIC penalizes extra short regimes more strongly than AIC.